# Phase 1: Baseline DP-SGD Implementation on MNIST
**Project:** Mitigating Clipping Bias in DP-SGD via Error Feedback (Dice-SGD)  
**Authors:** Priyanshu Agarwal & Sudiksha Singh

### Objective
Before implementing the custom Dice-SGD optimizer, we must establish a formal baseline using standard Differentially Private Stochastic Gradient Descent (DP-SGD). This notebook trains a simple CNN on the MNIST dataset using the Meta `opacus` library. It demonstrates the optimization degradation (clipping bias) caused by standard gradient clipping, serving as our control group.

In [ ]:
# Install required privacy and vision libraries
!pip install -q torch torchvision opacus

### 1. Environment Setup & Data Loading
We use the standard MNIST dataset. To balance gradient estimation reliability with computational speed, we use a batch size of `64`.

In [ ]:
import warnings
import torch
from torchvision import datasets, transforms
from opacus import PrivacyEngine
from torch import nn, optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# Suppress expected Opacus warnings for cleaner output during experimentation
warnings.filterwarnings("ignore", category=UserWarning)

# Configure device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Data transformation and loading
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

print("Downloading and preparing MNIST dataset...")
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)

### 2. Model Architecture
A standard, lightweight Convolutional Neural Network (CNN) suitable for 28x28 grayscale images. Note that for this baseline, standard pooling and linear layers are sufficient.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=8, stride=2, padding=3)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=4, stride=2)
        self.fc1 = nn.Linear(32 * 4 * 4, 32)
        self.fc2 = nn.Linear(32, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.max_pool2d(x, 2, 1)
        x = torch.relu(self.conv2(x))
        x = torch.max_pool2d(x, 2, 1)
        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleCNN().to(device)

### 3. Privacy Engine Integration & Training Loop
Here we wrap the standard PyTorch optimizer in Opacus's `PrivacyEngine`.
* **Clipping Norm ($C$):** `1.0` (Standard bound for sensitivity)
* **Noise Multiplier ($\sigma$):** `1.0`
* **Learning Rate:** `0.05`
* **Momentum:** `0.5` (Reduced to prevent noisy updates from compounding)

In [ ]:
# Optimizer and Loss setup
optimizer = optim.SGD(model.parameters(), lr=0.05, momentum=0.5)
criterion = nn.CrossEntropyLoss()

# Attach the Opacus Privacy Engine
privacy_engine = PrivacyEngine()
model, optimizer, train_loader = privacy_engine.make_private(
    module=model,
    optimizer=optimizer,
    data_loader=train_loader,
    noise_multiplier=1.0,
    max_grad_norm=1.0,
)

epochs = 5
test_accuracies = []

print(f"--- Starting Baseline DP-SGD Training ({epochs} Epochs) ---")
for epoch in range(epochs):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

    # Evaluation phase
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            output = model(images)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(labels.view_as(pred)).sum().item()

    accuracy = 100. * correct / len(test_loader.dataset)
    test_accuracies.append(accuracy)

    # Track the privacy budget using the RDP accountant
    epsilon = privacy_engine.get_epsilon(delta=1e-5)
    print(f"Epoch {epoch+1:02d} | Test Accuracy: {accuracy:.2f}% | Privacy Budget (\u03B5): {epsilon:.2f}")

### 4. Convergence Visualization
Visualizing the test accuracy trajectory. Notice the micro-oscillations caused by the continuous injection of Gaussian noise, demonstrating standard DP-SGD optimization dynamics.

In [ ]:
# Plotting the baseline results
plt.figure(figsize=(8, 5))
plt.plot(range(1, epochs+1), test_accuracies, marker='o', color='red', linestyle='--', linewidth=2)
plt.title('Baseline DP-SGD: Test Accuracy vs Epochs (MNIST)', fontsize=14)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.grid(True, linestyle=':', alpha=0.7)
plt.xticks(range(1, epochs+1))
plt.tight_layout()
plt.show()